In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv(r"C:\Users\User\Desktop\water\data\water_stats.csv")

In [6]:
# 1. Ορίζουμε τις βασικές παραμέτρους (αφαιρώντας τα διπλότυπα salinity & dissolved oxygen)
FEATURE_COLS = [
    'wtempV_°C', 
    'conductivity_of_solution_ mS_cm', 
    'oxidation_reduction_potential_ mV', 
    'turbidity_NTU', 
    'oxygen_saturation_%'
]

# Βεβαιώσου ότι η στήλη της ημερομηνίας είναι σε datetime μορφή
df['date_insert'] = pd.to_datetime(df['date_insert'])

In [7]:
# 1. Αντικατάσταση κωδικών σφάλματος (sentinel values) με NaN
df = df.replace([9999.01, 9999, -999], np.nan)

# 2. Φυσικά όρια ανά παράμετρο
physical_limits = {
    'wtempV_°C': (0.0, 45.0),
    'conductivity_of_solution_ mS_cm': (0.0, 80000.0), # Προσαρμοσμένο για uS/cm
    'oxidation_reduction_potential_ mV': (-1000.0, 1000.0),
    'turbidity_NTU': (0.0, 500.0),
    'oxygen_saturation_%': (0.0, 200.0)
}

# Εφαρμογή των φυσικών ορίων
for col, (min_val, max_val) in physical_limits.items():
    if col in df.columns:
        df.loc[(df[col] < min_val) | (df[col] > max_val), col] = np.nan

In [8]:
def clean_failing_sensors(group):
    # 1. Υπολογισμός Rolling Mean & Rolling Std (Παράθυρο 24 μετρήσεων ~ 1 ημέρα)
    roll_mean = group['oxygen_saturation_%'].rolling(window=24, min_periods=1).mean()
    roll_std = group['oxygen_saturation_%'].rolling(window=24, min_periods=1).std()
    
    # Kανόνας Α: Αν ο μέσος όρος ημέρας πέσει κάτω από 10%, ο αισθητήρας έχει πεθάνει/μπλοκάρει
    is_dead_zone = roll_mean < 10.0
    
    # Κανόνας Β: Ακραία spikes / θόρυβος (Std > 25% σε παράθυρο μιας ημέρας)
    is_noisy_zone = roll_std > 25.0
    
    # Εφαρμογή: Ακύρωση όσων εμπίπτουν στα παραπάνω
    group.loc[is_dead_zone | is_noisy_zone, 'oxygen_saturation_%'] = np.nan
    
    return group

# Εφαρμογή στον καθαρισμό
df_clean = df.groupby('sensor_id', group_keys=False).apply(clean_failing_sensors)

C:\Users\User\AppData\Local\Temp\ipykernel_5752\2886254624.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_clean = df.groupby('sensor_id', group_keys=False).apply(clean_failing_sensors)


In [9]:
# Αφαίρεση μεμονωμένων ακραίων spikes ανά αισθητήρα
def remove_extreme_iqr(group):
    for col in FEATURE_COLS:
        if col in group.columns and group[col].notna().sum() > 10:
            Q1 = group[col].quantile(0.25)
            Q3 = group[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 3 * IQR
            upper_bound = Q3 + 3 * IQR
            group.loc[(group[col] < lower_bound) | (group[col] > upper_bound), col] = np.nan
    return group

df_clean = df_clean.groupby('sensor_id', group_keys=False).apply(remove_extreme_iqr)

C:\Users\User\AppData\Local\Temp\ipykernel_5752\2471801742.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_clean = df_clean.groupby('sensor_id', group_keys=False).apply(remove_extreme_iqr)


In [10]:
# 1. Ταξινόμηση ανά αισθητήρα και χρόνο
df_clean = df_clean.sort_values(['sensor_id', 'date_insert'])

# 2. Γραμμική παρεμβολή ΜΟΝΟ για μικρά κενά (έως 12 μετρήσεις / λίγες ώρες)
df_clean[FEATURE_COLS] = df_clean.groupby('sensor_id')[FEATURE_COLS].transform(
    lambda group: group.interpolate(method='linear', limit=12)
)

# 3. Πετάμε τα εναπομείναντα NaNs (τα μεγάλα νεκρά διαστήματα μηνών)
df_ready = df_clean.dropna(subset=FEATURE_COLS).copy()

# Έλεγχος αποτελεσμάτων
print(f"Αρχικές εγγραφές dataset: {len(df)}")
print(f"Καθαρές εγγραφές για PCA / ML: {len(df_ready)}")
print("\n--- Remaining Missing Values in Clean Dataset ---")
print(df_ready[FEATURE_COLS].isna().sum())

Αρχικές εγγραφές dataset: 34146
Καθαρές εγγραφές για PCA / ML: 17957

--- Remaining Missing Values in Clean Dataset ---
wtempV_°C                            0
conductivity_of_solution_ mS_cm      0
oxidation_reduction_potential_ mV    0
turbidity_NTU                        0
oxygen_saturation_%                  0
dtype: int64


In [11]:
df_ready.to_csv('C:/Users/User/Desktop/water/data/clean_water_stats.csv')